In [2]:
import gurobipy as gp
from gurobipy import GRB, nlfunc
from utils.data import *

In [ ]:
start= dt.datetime(year=2000, month=1, day=1)
end= dt.datetime(year=2023, month=1, day=1)
all_stocks = ['AAPL', "GOOG", "NVDA", "CL", "BA", "NKE", "DB", ""]

n = len(all_stocks)

mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)
#Sigma_XY = Sigma[]

r = 0.05
T = 90
S = [yf.Ticker(s).history(period="1d") for s in all_stocks]

In [67]:
m = gp.Model()

S = [yf.Ticker(s).history(period="1d").Close.to_list()[0] for s in all_stocks]
sigma = [Sigma[i, i] for i in range(n)]

w = m.addMVar(n, lb=0, name="weight")

mu_d = m.addMVar(n, name="expectation")
Delta = m.addMVar(n, lb=0.1, ub=.9, name="Delta")
Phi_inv = m.addMVar(n, name="phi of inverse CDF")
phi = m.addMVar(n, name="phi")
K = m.addMVar(n, lb=0, name="strike")
P = m.addMVar(n, lb=0, name="put price")
d1 = m.addMVar(n, lb=-GRB.INFINITY, name="d1")
d2 = m.addMVar(n, lb=-GRB.INFINITY, name="d2")

t = m.addVar()
var_norm = m.addVar()

m.addConstr(gp.quicksum([wi for wi in w]) == 1)
m.addConstr(var_norm**2 == w.T @ Sigma @ w)
m.addConstr(mu@w <= var_norm * t)
m.addConstrs(mu_d[i] == mu[i] + sigma[i]*(Delta[i]*Phi_inv[i] + phi[i])\
                            for i in range(n))
m.addConstrs(Phi_inv[i] == -4/10*nlfunc.log(1/(Delta[i]/2+1/2)-1) for i in range(n))
m.addConstrs(phi[i] == nlfunc.exp(-Phi_inv[i]**2/2)/np.sqrt(2*np.pi) for i in range(n))
m.addConstrs(K[i] == sigma[i]*Phi_inv[i] + S[i] for i in range(n))

m.addConstrs(d1[i] == (nlfunc.log(S[i]/K[i])+T*(r+sigma[i]**2/2))/(sigma[i]*np.sqrt(T)) for i in range(n))
m.addConstrs(d2[i] == d1[i] - sigma[i]*np.sqrt(T) for i in range(n))

#m.addConstrs(P[i] == K[i]*np.exp(-r*T)/(1+nlfunc.exp(2.5*d2[i])) - S[i]/(1+nlfunc.exp(2.5*d1[i])) for i in range(n))

m.setObjective(gp.quicksum([mu[i] + sigma[i]*(Delta[i]*Phi_inv[i] + phi[i]) - P[i]\
                            for i in range(n)]), GRB.MAXIMIZE)

m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Fedora Linux 42 (Workstation Edition)")

CPU model: AMD Ryzen 9 5900X 12-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 15 rows, 65 columns and 35 nonzeros
Model fingerprint: 0x3947830a
Model has 7 quadratic objective terms
Model has 9 quadratic constraints
Model has 21 general nonlinear constraints (28 nonlinear terms)
Variable types: 65 continuous, 0 integer (0 binary)
Coefficient statistics:
  Matrix range     [8e-03, 1e+00]
  QMatrix range    [3e-04, 1e+00]
  QLMatrix range   [3e-03, 1e+00]
  Objective range  [8e-03, 1e+00]
  QObjective range [2e-02, 2e-01]
  Bounds range     [1e-01, 9e-01]
  RHS range        [8e-02, 3e+02]
  QRHS range       [3e-03, 1e-01]
Presolve model has 21 nlconstr
Added 66 variables to disaggregate expressions.
Presolve removed 7 rows and 21 columns
Presolve time: 0.00s
Presolved: 431 rows, 148 co

In [68]:
weights = np.zeros(n)

var = m.getVars()

weights = [v.X for v in var if "weight" in v.VarName]

deltas = [v.X for v in var if "Delta" in v.VarName]

weights, deltas

([0.14589263892019458,
  0.13390378452460383,
  0.19391453526563576,
  0.1184163662256228,
  0.13479768605430673,
  0.12540092931258925,
  0.14767405969704697],
 [0.9, 0.9, 0.9, 0.9, 0.9, 0.9, 0.9])

In [69]:
import plotly.express as px

px.line(benchmark(all_stocks, weights, "SPY", end, dt.datetime.today()))

In [56]:
px.line(benchmark(all_stocks, np.ones(len(all_stocks))/len(all_stocks), "SPY", end, dt.datetime.today()))

In [31]:
sigma = [np.std(get_hist(s, start, end).Open.to_numpy()) for s in all_stocks]

sigma

[np.float64(41.13958930297493),
 np.float64(33.92487904583442),
 np.float64(5.754463949391831),
 np.float64(19.614596993522884),
 np.float64(94.9451142725771),
 np.float64(37.677191413275544),
 np.float64(18.383465051812426)]